In [1]:
import nest_asyncio
nest_asyncio.apply()

In [2]:
!pip install flask flask-cors scipy -q

In [3]:
!pip install fastapi uvicorn pyngrok nest-asyncio joblib xgboost scipy httpx python-multipart -q

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from google.colab import files
uploaded = files.upload()

import shutil
shutil.move(list(uploaded.keys())[0], '/content/courier_guard.html')
print("HTML uploaded")

Saving courier_guard (1) copy 3.html to courier_guard (1) copy 3.html
HTML uploaded


In [ ]:
# Upload the heatmap HTML file
from google.colab import files
import shutil

uploaded_hm = files.upload()
shutil.move(list(uploaded_hm.keys())[0], '/content/courier_guard_heatmap.html')
print('Heatmap HTML uploaded ✅')


In [ ]:
# Upload the admin dashboard HTML file
from google.colab import files
import shutil

uploaded_db = files.upload()
shutil.move(list(uploaded_db.keys())[0], '/content/admin_dashboard.html')
print('Dashboard HTML uploaded ✅')


In [ ]:
# Upload dashboard HTML
from google.colab import files
import shutil
uploaded_db = files.upload()
shutil.move(list(uploaded_db.keys())[0], '/content/admin_dashboard.html')
print('Dashboard uploaded ✅')


In [ ]:
import math as _math

def stull_wbgt(temp_c: float, humidity: float, sun_condition: str = "full_sun") -> float:
    """
    Wet-bulb approximation — Stull (2011, J Appl Meteorol Climatol 50:2267-2269).
    Outdoor WBGT = 0.7 × Tw + 0.2 × Tg + 0.1 × Tdb   (ISO 7243)
    Unifies heat model with HTML frontend approxWBGT().
    """
    rh = humidity
    tw = (temp_c * _math.atan(0.151977 * (rh + 8.313659)**0.5)
          + _math.atan(temp_c + rh)
          - _math.atan(rh - 1.676331)
          + 0.00391838 * rh**1.5 * _math.atan(0.023101 * rh)
          - 4.686035)
    tg = temp_c + (10 if sun_condition == "full_sun" else 4 if sun_condition == "partly_cloudy" else 0)
    return round(0.7 * tw + 0.2 * tg + 0.1 * temp_c, 1)


import os, nest_asyncio, asyncio, uvicorn, numpy as np, pandas as pd
import scipy.fftpack, joblib
from fastapi import FastAPI, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, HTMLResponse
from pydantic import BaseModel
from pyngrok import ngrok

nest_asyncio.apply()

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])

@app.middleware("http")
async def skip_ngrok_warning(request: Request, call_next):
    response = await call_next(request)
    response.headers["ngrok-skip-browser-warning"] = "true"
    response.headers["Access-Control-Allow-Origin"] = "*"
    return response
SAVE_PATH = '/content/drive/MyDrive/'

print("Loading models...")
har_model    = joblib.load(os.path.join(SAVE_PATH, 'har_master_courier_loso.pkl'))
weee_static  = joblib.load(os.path.join(SAVE_PATH, 'rf_static_final_no_hr.pkl'))
weee_dynamic = joblib.load(os.path.join(SAVE_PATH, 'rf_dyn_final_no_hr.pkl'))


if hasattr(har_model, 'feature_names_in_'):
    HAR_COLS = list(har_model.feature_names_in_)
else:
    HAR_COLS = har_model.get_booster().feature_names
print(f" HAR model    : {len(HAR_COLS)} features")
print(f"   {HAR_COLS}")

# ── WEEE static — handle dict (new) or RF object (old) ───────────────────────
if isinstance(weee_static, dict):
    STATIC_MET = max(float(weee_static['mean_MET']), 1.0)    
    print(f" WEEE static  : population mean MET = {STATIC_MET:.4f}")
else:
    STATIC_MET = 1.0   
    print(f"  WEEE static  : old RF model found, using fallback MET = {STATIC_MET}")

# ── WEEE dynamic feature columns — read directly from saved model ─────────────
if hasattr(weee_dynamic, 'feature_names_in_'):
    WEEE_COLS = list(weee_dynamic.feature_names_in_)
    print(f" WEEE dynamic : {len(WEEE_COLS)} features")
    print(f"   {WEEE_COLS}")
else:
    WEEE_COLS = [
        'acc_mag_mean','acc_mag_std','acc_mag_max','acc_mag_min',
        'acc_mag_p2p','acc_mag_iqr','acc_x_mean','acc_y_mean','acc_z_mean',
        'acc_x_std','acc_y_std','acc_z_std','acc_mag_energy','acc_sma',
        'acc_mag_zcr','acc_xy_corr','acc_xz_corr','acc_yz_corr',
    ]
    print(f"⚠  WEEE dynamic : no feature_names_in_, using fallback list")

print("All models ready\n")

#---------weather api + rest schedule

import httpx
import uuid
from pathlib import Path
from datetime import datetime
from fastapi import UploadFile, File, Form

OPENWEATHER_KEY = " open weather key "

PHOTO_DIR = Path("/content/delivery_photos")
PHOTO_DIR.mkdir(exist_ok=True)

#NIOSH 2017-127 Work/Rest Table  
NIOSH_TABLE = {
    90:  {"light": None,        "moderate": None,        "heavy": None},
    91:  {"light": None,        "moderate": None,        "heavy": None},
    92:  {"light": None,        "moderate": None,        "heavy": None},
    93:  {"light": None,        "moderate": None,        "heavy": None},
    94:  {"light": None,        "moderate": None,        "heavy": None},
    95:  {"light": None,        "moderate": None,        "heavy": (45, 15)},
    96:  {"light": None,        "moderate": None,        "heavy": (45, 15)},
    97:  {"light": None,        "moderate": None,        "heavy": (40, 20)},
    98:  {"light": None,        "moderate": None,        "heavy": (35, 25)},
    99:  {"light": None,        "moderate": None,        "heavy": (35, 25)},
    100: {"light": None,        "moderate": (45, 15),    "heavy": (30, 30)},
    101: {"light": None,        "moderate": (40, 20),    "heavy": (30, 30)},
    102: {"light": None,        "moderate": (35, 25),    "heavy": (25, 35)},
    103: {"light": None,        "moderate": (30, 30),    "heavy": (20, 40)},
    104: {"light": None,        "moderate": (30, 30),    "heavy": (20, 40)},
    105: {"light": None,        "moderate": (25, 35),    "heavy": (15, 45)},
    106: {"light": (45, 15),    "moderate": (20, 40),    "heavy": "caution"},
    107: {"light": (40, 20),    "moderate": (15, 45),    "heavy": "caution"},
    108: {"light": (35, 25),    "moderate": "caution",   "heavy": "caution"},
    109: {"light": (30, 30),    "moderate": "caution",   "heavy": "caution"},
    110: {"light": (15, 45),    "moderate": "caution",   "heavy": "caution"},
    111: {"light": "caution",   "moderate": "caution",   "heavy": "caution"},
    112: {"light": "caution",   "moderate": "caution",   "heavy": "caution"},
}

def niosh_adj_temp(temp_c: float, humidity_pct: float, sun_condition: str = "full_sun") -> float:
    """
    Apply NIOSH temperature adjustments for sun exposure and humidity.
    Returns adjusted temperature in °F.
    """
    temp_f = temp_c * 9 / 5 + 32
    # Sun adjustment
    if sun_condition == "full_sun":
        temp_f += 13
    elif sun_condition == "partly_cloudy":
        temp_f += 7
    # Humidity adjustment
    if humidity_pct >= 60:
        temp_f += 9
    elif humidity_pct >= 50:
        temp_f += 6
    elif humidity_pct >= 40:
        temp_f += 3
    return round(temp_f, 1)

def niosh_lookup(adj_temp_f: float, intensity: str = "moderate") -> dict:
    """
    Look up NIOSH work/rest schedule.
    Returns {type: 'normal'|'restricted'|'caution', work_min, rest_min}
    """
    t = max(90, min(112, round(adj_temp_f)))
    row = NIOSH_TABLE.get(t)
    if not row:
        return {"type": "normal", "work_min": None, "rest_min": None}
    val = row.get(intensity)
    if val is None:
        return {"type": "normal", "work_min": None, "rest_min": None}
    if val == "caution":
        return {"type": "caution", "work_min": None, "rest_min": None}
    return {"type": "restricted", "work_min": val[0], "rest_min": val[1]}


# ── Pydantic models ────────────────────────────────────────────────────────────
class WeatherRequest(BaseModel):
    lat: float = 37.5665           
    lon: float = 126.9780
    sun_condition: str = "full_sun"   

class DeliveryEstimate(BaseModel):
    id: str
    addr: str
    floor: int
    has_elevator: bool
    pkg_weight_kg: float
    pkg_count: int
    status: str = "pending"

class RestRequest(BaseModel):
    deliveries: list[DeliveryEstimate]
    completed_count: int = 0
    continuous_work_min: float = 0.0     
    continuous_met_min: float = 0.0     
    courier_weight_kg: float = 75.0
    temp_c: float = 27.0
    humidity: float = 65.0
    sun_condition: str = "full_sun"

 
def stair_carry_met(load_kg: float) -> float:
    """
    MET for stair climbing while carrying a load.
    Ainsworth 2011 Compendium exact codes:
      17026: 1–15 lb  (<6.8 kg)      upstairs → 5.0 MET
      17027: 16–24 lb (7.3–10.9 kg)  upstairs → 6.0 MET
      17028: 25–49 lb (11.3–22.2 kg) upstairs → 8.0 MET
      17029: 50–74 lb (22.7–33.6 kg) upstairs → 10.0 MET
      17030: >74 lb   (>33.6 kg)     upstairs → 12.0 MET
    """
    lbs = load_kg * 2.205
    if lbs <= 15:  return 5.0
    if lbs <= 24:  return 6.0
    if lbs <= 49:  return 8.0
    if lbs <= 74:  return 10.0
    return 12.0


def flat_carry_met(load_kg: float) -> float:
    """
    MET for flat-ground walking while carrying a load. Ainsworth 2011:
      11795: <25 lb (<11.3 kg) slow walk → 3.5 MET
      11800: <25 lb moderate carry       → 4.5 MET
      11820: 25–49 lb (>11.3 kg) walk carrying load → 5.0 MET
    """
    if load_kg < 6.8:  return 3.5
    if load_kg < 11.3: return 4.5
    return 5.0


def delivery_phases(floor: int, has_elevator: bool,
                    pkg_weight_kg: float, pkg_count: int):
    """
    Phase-based model of the courier's actual delivery workflow.

    Elevator buildings (has_elevator=True):
      Courier pulls loaded cart to elevator, rides up pressing each
      floor button. On each floor, carries only that 호수's boxes to
      the door and returns to elevator. Cart stays in lobby.
        Phase 1: push cart flat to elevator (code 17100 → 4.0 MET)
        Phase 2: elevator ride up (static → 1.3 MET)
        Phase 3: walk to unit carrying boxes (codes 11795/11800/11820 by load)
        Phase 4: return empty to elevator (3.5 MET)
        Phase 5: elevator ride down (1.3 MET)

    No-elevator buildings (has_elevator=False, floor > 1):
      Courier pulls cart to building entrance, then carries boxes for
      the lowest floor first up the stairs, delivers, descends empty,
      picks up next load, repeats floor by floor.
        Phase 1: pull cart to building entrance (4.0 MET)
        Phase 2: stair climb with boxes (codes 17026–17030 by load)
        Phase 3: walk to unit on that floor (flat carry MET)
        Phase 4: descend empty (code 17070 → 3.5 MET)

    Returns (phases, total_dur_min, avg_met) where
      phases = list of (met, dur_min, label).
    """
    load_kg     = pkg_weight_kg * pkg_count
    floor_above = max(floor - 1, 0)
    phases: list[tuple[float, float, str]] = []

    if floor == 1:
        dur = max(2.0 + 0.5 * pkg_count, 2.0)
        phases.append((flat_carry_met(load_kg), dur, "평지 이동(하중)"))

    elif has_elevator:
        # 1. Push loaded cart from truck to elevator (~1.5 min)
        phases.append((4.0, 1.5, "카트 이동(평지→엘베)"))
        # 2. Elevator up: 0.5 min wait + 0.3 min/floor
        elev_dur = 0.5 + 0.3 * floor
        phases.append((1.3, elev_dur, "엘리베이터(상행)"))
        # 3. Carry boxes from elevator to unit (~1.5 + 0.3/pkg min)
        phases.append((flat_carry_met(load_kg),
                       1.5 + 0.3 * pkg_count, "호수까지(하중)"))
        # 4. Return empty to elevator (~1.0 min)
        phases.append((3.5, 1.0, "엘리베이터 복귀(빈손)"))
        # 5. Elevator down (~80% of up time, fewer stops)
        phases.append((1.3, elev_dur * 0.8, "엘리베이터(하행)"))

    else:
        # 1. Pull loaded cart to building entrance (~1.5 min)
        phases.append((4.0, 1.5, "카트 이동(건물 입구)"))
        # 2. Stair climb carrying boxes: 1.5 min/flight
        stair_up_dur = 1.5 * floor_above
        phases.append((stair_carry_met(load_kg),
                       stair_up_dur, "계단 오르기(하중)"))
        # 3. Walk to unit on that floor (~1.5 + 0.3/pkg min)
        phases.append((flat_carry_met(load_kg),
                       1.5 + 0.3 * pkg_count, "호수까지(하중)"))
        # 4. Descend stairs empty: 1.0 min/flight — code 17070: 3.5 MET
        stair_down_dur = 1.0 * floor_above
        phases.append((3.5, stair_down_dur, "계단 내려오기(빈손)"))

    total_dur = max(sum(d for _, d, _ in phases), 2.0)
    avg_met   = (sum(m * d for m, d, _ in phases) / total_dur
                 if total_dur > 0 else 3.5)
    return phases, total_dur, avg_met


def estimate_delivery_met(floor: int, has_elevator: bool,
                           pkg_weight_kg: float, pkg_count: int) -> float:
    """
    Phase-based average MET for a single delivery.
    Uses exact 2011 Compendium codes:
      - Stair carry with load: codes 17026–17030 (5.0–12.0 MET by load)
      - Stair descend unloaded: code 17070 (3.5 MET)
      - Elevator riding: 1.3 MET (static)
      - Cart push/pull flat: code 17100 (4.0 MET)
      - Flat walk with load: codes 11795/11800/11820 (3.5–5.0 MET)
    """
    _, _, avg_met = delivery_phases(floor, has_elevator, pkg_weight_kg, pkg_count)
    return round(min(avg_met, 12.0), 2)


def estimate_delivery_duration_min(floor: int, has_elevator: bool,
                                    pkg_count: int,
                                    pkg_weight_kg: float = 5.0) -> float:
    """Estimate delivery time in minutes based on phase durations."""
    _, total_dur, _ = delivery_phases(floor, has_elevator, pkg_weight_kg, pkg_count)
    return round(total_dur, 1)


#weather

@app.post("/weather")
async def get_weather(req: WeatherRequest):
    """
    Fetch real-time weather and compute NIOSH heat stress level.
    """
    try:
        async with httpx.AsyncClient(timeout=6.0) as client:
            r = await client.get(
                "https://api.openweathermap.org/data/2.5/weather",
                params={
                    "lat": req.lat, "lon": req.lon,
                    "appid": OPENWEATHER_KEY, "units": "metric"
                }
            )
        if r.status_code != 200:
            raise HTTPException(502, f"Weather API returned {r.status_code}")
        data = r.json()
    except httpx.TimeoutException:
        raise HTTPException(504, "Weather API timeout")

    temp_c   = data["main"]["temp"]
    humidity = data["main"]["humidity"]
    adj_f    = niosh_adj_temp(temp_c, humidity, req.sun_condition)

 
    levels = {}
    for intensity in ("light", "moderate", "heavy"):
        n = niosh_lookup(adj_f, intensity)
        if n["type"] == "caution":
            sched = "극고위험 — 작업 즉시 중단"
        elif n["type"] == "restricted":
            sched = f"작업 {n['work_min']}분 → 휴식 {n['rest_min']}분"
        else:
            sched = "정상 (제한 없음)"
        levels[intensity] = {**n, "schedule_text": sched}

    
    mod = levels["moderate"]
    hvy = levels["heavy"]
    if mod["type"] == "caution" or hvy["type"] == "caution":
        risk_level = "극고위험"; risk_color = "red"
    elif hvy["type"] == "restricted":
        risk_level = "열환경 위험"; risk_color = "orange"
    elif mod["type"] == "restricted":
        risk_level = "열환경 주의"; risk_color = "yellow"
    else:
        risk_level = "✅정상"; risk_color = "green"

    return {
        "temp_c":          round(temp_c, 1),
        "temp_f":          round(temp_c * 9 / 5 + 32, 1),
        "humidity":        humidity,
        "adjusted_temp_f": adj_f,
        "sun_condition":   req.sun_condition,
        "description":     data["weather"][0]["description"],
        "city":            data.get("name", ""),
        "wbgt":            stull_wbgt(temp_c, humidity, req.sun_condition),
        "risk_level":      risk_level,
        "risk_color":      risk_color,
        "niosh_light":     levels["light"],
        "niosh_moderate":  levels["moderate"],
        "niosh_heavy":     levels["heavy"],
    }


 
@app.post("/rest_recommendation")
def rest_recommendation(req: RestRequest):
    completed = req.completed_count
    pending   = req.deliveries[completed:]

    if not pending:
        return {"rest_points": [], "message": "모든 배송 완료", "total_rest_min": 0}

    adj_f = niosh_adj_temp(req.temp_c, req.humidity, req.sun_condition)
    mets  = [
        estimate_delivery_met(d.floor, d.has_elevator, d.pkg_weight_kg, d.pkg_count)
        for d in pending
    ]
    avg_met   = sum(mets) / len(mets)
    intensity = "moderate" if avg_met < 5.0 else "heavy"
    niosh     = niosh_lookup(adj_f, intensity)
 
    heat_alert = False
    if niosh["type"] == "caution":
        work_limit_min = 10;  rest_dur_min = 30;  heat_alert = True
    elif niosh["type"] == "restricted":
        work_limit_min = niosh["work_min"]
        rest_dur_min   = niosh["rest_min"]
        heat_alert     = True
    else: 
        work_limit_min = 90 if intensity == "moderate" else 60
        rest_dur_min   = 10 if intensity == "moderate" else 15
 
    MET_THRESHOLD = 350 if intensity == "moderate" else 240
 
    rest_points  = []
    cum_work_min = req.continuous_work_min
    cum_met_min  = req.continuous_met_min

    for i, d in enumerate(pending):
        dur_min   = estimate_delivery_duration_min(d.floor, d.has_elevator, d.pkg_count, d.pkg_weight_kg)
        met       = mets[i]
        cum_work_min += dur_min
        cum_met_min  += met * dur_min

        needs_rest = (cum_work_min >= work_limit_min or cum_met_min >= MET_THRESHOLD)

        if needs_rest and i < len(pending) - 1:
            reasons = []
            if heat_alert:
                reasons.append(f" 열환경 위험 ({req.temp_c:.0f}°C, 습도 {req.humidity:.0f}%, 체감 {adj_f:.0f}°F)")
            if cum_met_min >= MET_THRESHOLD:
                reasons.append(f"누적 대사 부하 {cum_met_min:.0f} MET·분 초과 (임계값 {MET_THRESHOLD})")
            if cum_work_min >= work_limit_min:
                reasons.append(f"연속 작업 {cum_work_min:.0f}분 도달 (NIOSH 제한 {work_limit_min}분)")

            rest_points.append({
                "after_delivery_index": completed + i,
                "after_delivery_addr":  d.addr,
                "rest_duration_min":    rest_dur_min,
                "intensity":            intensity,
                "heat_alert":           heat_alert,
                "cumulative_work_min":  round(cum_work_min, 1),
                "cumulative_met_min":   round(cum_met_min, 1),
                "adjusted_temp_f":      adj_f,
                "reasons":              reasons,
            })
            # Reset counters after rest
            cum_work_min = 0
            cum_met_min  = 0

    total_rest = sum(rp["rest_duration_min"] for rp in rest_points)
    alert_str  = "열환경 위험 — " if heat_alert else ""
    return {
        "rest_points":     rest_points,
        "intensity":       intensity,
        "avg_met":         round(avg_met, 2),
        "heat_alert":      heat_alert,
        "niosh_schedule":  niosh,
        "adjusted_temp_f": adj_f,
        "total_rest_min":  total_rest,
        "message":         f"{alert_str}{len(rest_points)}회 휴식 권장 (총 {total_rest}분)",
    }


# delivery/photo 
@app.post("/delivery/photo")
async def upload_delivery_photo(
    delivery_id: str    = Form(...),
    worker_id:   str    = Form(default="unknown"),
    photo:       UploadFile = File(...),
):
    """
    Accept a proof-of-delivery photo.
    Saves to /content/delivery_photos/{date}/{delivery_id}_{ts}.jpg
    Returns metadata that the client stores alongside the delivery record.
    """
    date_str  = datetime.now().strftime("%Y%m%d")
    day_dir   = PHOTO_DIR / date_str
    day_dir.mkdir(exist_ok=True)

    ext      = (photo.filename or "photo.jpg").rsplit(".", 1)[-1].lower()
    ext      = ext if ext in ("jpg","jpeg","png","webp","heic") else "jpg"
    ts_str   = datetime.now().strftime("%H%M%S")
    uid      = uuid.uuid4().hex[:6]
    filename = f"{delivery_id}_{ts_str}_{uid}.{ext}"
    fpath    = day_dir / filename

    contents = await photo.read()
    if len(contents) > 20 * 1024 * 1024:   # 20 MB guard
        raise HTTPException(413, "Photo too large (max 20 MB)")

    with open(fpath, "wb") as f:
        f.write(contents)

    return {
        "success":     True,
        "delivery_id": delivery_id,
        "worker_id":   worker_id,
        "filename":    filename,
        "path":        str(fpath),
        "size_bytes":  len(contents),
        "timestamp":   datetime.now().isoformat(),
        "date":        date_str,
    }



CLASS_INFO = {
    0: {"name":"Standing/Sitting","korean":"대기/앉기",  "domain":"static",  "icon":"🧍"},
    1: {"name":"Elevator",        "korean":"엘리베이터", "domain":"static",  "icon":"🛗"},
    2: {"name":"Stairs UP",       "korean":"계단 오르기","domain":"dynamic", "icon":"⬆️"},
    3: {"name":"Stairs DOWN",     "korean":"계단 내리기","domain":"dynamic", "icon":"⬇️"},
    4: {"name":"Walking",         "korean":"평지 걷기",  "domain":"dynamic", "icon":"🚶"},
    5: {"name":"Running",         "korean":"뛰기",       "domain":"dynamic", "icon":"🏃"},
    6: {"name":"Cycling",         "korean":"자전거",     "domain":"dynamic", "icon":"🚲"},
}

 

def extract(ax, ay, az, gx, gy, gz, sample_hz=60.0):
    def f(x, y, z, p):
        x,y,z = np.array(x,float), np.array(y,float), np.array(z,float)
        mag   = np.sqrt(x**2+y**2+z**2)
        if p == 'acc' and mag.mean() > 5.0:
            x = x - x.mean()
            y = y - y.mean()
            z = z - z.mean()
            mag = np.sqrt(x**2+y**2+z**2)
        mc    = mag - mag.mean()
        fft_v = np.abs(scipy.fftpack.fft(mag))
        freqs = scipy.fftpack.fftfreq(len(mag), d=1/sample_hz)
        pidx  = 1+np.argmax(fft_v[1:]) if len(fft_v)>1 else 0
        fs    = fft_v.sum()+1e-9
        # z-axis FFT for vertical-cadence dominant frequency
        fft_z = np.abs(scipy.fftpack.fft(z - z.mean()))
        z_pidx = 1+np.argmax(fft_z[1:]) if len(fft_z)>1 else 0
        # skew / kurt helpers
        def _skew(a):
            m,s = a.mean(), a.std()
            return float(((a-m)**3).mean() / (s**3 + 1e-9))
        def _kurt(a):
            m,s = a.mean(), a.std()
            return float(((a-m)**4).mean() / (s**4 + 1e-9) - 3.0)
        # z-axis pos/neg peak imbalance
        z_pos = z[z > 0]
        z_neg = z[z < 0]
        z_pos_mean = float(z_pos.mean()) if len(z_pos) else 0.0
        z_neg_mean = float(z_neg.mean()) if len(z_neg) else 0.0
        dt = 1.0/sample_hz
        return {
            f'{p}_x_mean':float(x.mean()),   f'{p}_y_mean':float(y.mean()),
            f'{p}_z_mean':float(z.mean()),   f'{p}_x_std': float(x.std()),
            f'{p}_y_std': float(y.std()),    f'{p}_z_std': float(z.std()),
            f'{p}_mag_mean':float(mag.mean()),f'{p}_mag_std':float(mag.std()),
            f'{p}_mag_p2p': float(np.ptp(mag)),
            f'{p}_mag_max': float(mag.max()), f'{p}_mag_min':float(mag.min()),
            f'{p}_mag_iqr': float(np.percentile(mag,75)-np.percentile(mag,25)),
            f'{p}_mag_energy':float((mag**2).mean()),
            f'{p}_sma':   float((np.abs(x)+np.abs(y)+np.abs(z)).mean()),
            f'{p}_mag_75th':float(np.percentile(mag,75)),
            f'{p}_mag_25th':float(np.percentile(mag,25)),
            f'{p}_zcr':   float(((mc[:-1]*mc[1:])<0).sum()/len(mag)),
            f'{p}_mag_zcr':float(((mc[:-1]*mc[1:])<0).sum()/len(mag)),
            f'{p}_dominant_freq':   float(abs(freqs[pidx])),
            f'{p}_spectral_entropy':float(-np.sum((fft_v/fs)*np.log(fft_v/fs+1e-9))),
            f'{p}_xy_corr':float(np.corrcoef(x,y)[0,1]) if len(x)>1 else 0.0,
            f'{p}_xz_corr':float(np.corrcoef(x,z)[0,1]) if len(x)>1 else 0.0,
            f'{p}_yz_corr':float(np.corrcoef(y,z)[0,1]) if len(y)>1 else 0.0,
            # ── Stair-discriminating features ──────────────────────────────
            f'{p}_z_drift':     float(np.sum(z) * dt),
            f'{p}_z_drift_abs': float(abs(np.sum(z) * dt)),
            f'{p}_z_skew':      _skew(z),
            f'{p}_z_kurt':      _kurt(z),
            f'{p}_mag_skew':    _skew(mag),
            f'{p}_mag_kurt':    _kurt(mag),
            f'{p}_z_pos_mean':  z_pos_mean,
            f'{p}_z_neg_mean':  z_neg_mean,
            f'{p}_z_peak_asym': z_pos_mean + z_neg_mean,
            f'{p}_z_dominant_freq': float(abs(freqs[z_pidx])),
        }
    out = {}
    out.update(f(ax,ay,az,'acc'))
    out.update(f(gx,gy,gz,'gyro'))
    return out

# ── startup self-check: confirm extract() covers every feature the HAR model wants ──
try:
    _chk = extract([0.1]*64,[0.1]*64,[9.8]*64,[0.01]*64,[0.01]*64,[0.01]*64, sample_hz=60.0)
    _chk['carrying_load'] = 0
    _missing_har = [c for c in HAR_COLS if c not in _chk]
    print(f"✓ extract() covers all {len(HAR_COLS)} HAR features" if not _missing_har
          else f"⚠ HAR features NOT produced by extract(): {_missing_har}")
except Exception as _e:
    print(f"⚠ self-check skipped: {_e}")

class IMUWindow(BaseModel):
    acc_x:list[float]; acc_y:list[float]; acc_z:list[float]
    gyro_x:list[float]; gyro_y:list[float]; gyro_z:list[float]
    carrying_load:int=0
    courier_weight_kg:float=75.0
    load_kg:float=0.0
    sample_hz:float=60.0

@app.get("/health")
def health():
    return {"status":"ok","models_loaded": har_model is not None}


@app.post("/predict")
def predict(w: IMUWindow):
    n = min(len(w.acc_x), len(w.gyro_x))
    if n < 10:
        raise HTTPException(400, f"Too few samples: {n}")
    try:
        feats = extract(w.acc_x, w.acc_y, w.acc_z,
                        w.gyro_x, w.gyro_y, w.gyro_z, sample_hz=w.sample_hz)
        feats['carrying_load'] = int(w.carrying_load)

        # HAR — columns come from har_model.feature_names_in_ (set at startup)
        X_har = pd.DataFrame([{col: feats.get(col, 0.0) for col in HAR_COLS}])[HAR_COLS]
        cls   = int(har_model.predict(X_har)[0])
        info  = CLASS_INFO.get(cls, CLASS_INFO[4])

        # WEEE
        if info['domain'] == 'static':
            met = STATIC_MET
        else:
            X_weee = pd.DataFrame([{col: feats.get(col, 0.0) for col in WEEE_COLS}])[WEEE_COLS]
            met    = float(weee_dynamic.predict(X_weee)[0])
            met    = max(met, 2.0)  # Compendium minimum for dynamic activity (walking, code 17151: 2.0 MET)

        load_used   = w.load_kg if w.carrying_load else 0.0
        cload       = (1 + 0.5 * load_used / w.courier_weight_kg) if load_used > 0 else 1.0
        energy_j    = met * cload * w.courier_weight_kg * (2.0/3600.0) * 4184

        return {
            "activity_class":  cls,
            "activity_name":   info["name"],
            "activity_korean": info["korean"],
            "activity_icon":   info["icon"],
            "domain":          info["domain"],
            "met":             round(met, 3),
            "energy_joules":   round(energy_j, 2),
            "energy_kcal":     round(energy_j/4184, 5),
        }
    except Exception as e:
        traceback.print_exc()
        raise HTTPException(500, f"{type(e).__name__}: {str(e)}")


@app.post("/predict_fast")
def predict_fast(w: IMUWindow):
    return predict(w)

HTML_PAGE = open('/content/courier_guard.html').read() if os.path.exists('/content/courier_guard.html') else "<h1>Upload courier_guard.html to /content/</h1>"

@app.get("/", response_class=HTMLResponse)
def index():
    return HTMLResponse(content=HTML_PAGE)




HEATMAP_PAGE = open('/content/courier_guard_heatmap.html').read() if os.path.exists('/content/courier_guard_heatmap.html') else "<h1>Upload courier_guard_heatmap.html to /content/</h1>"

@app.get("/heatmap", response_class=HTMLResponse)
def heatmap():
    return HTMLResponse(content=HEATMAP_PAGE)


DASHBOARD_PAGE = open('/content/admin_dashboard.html').read() if os.path.exists('/content/admin_dashboard.html') else "<h1>Upload admin_dashboard.html</h1>"

@app.get("/dashboard", response_class=HTMLResponse)
def dashboard():
    return HTMLResponse(content=DASHBOARD_PAGE)


#records
import json as _json
import traceback  # used by /predict's error handler

RECORDS_PATH = os.path.join(SAVE_PATH, 'courier_daily_records.json')

def _load_records():
    try:
        with open(RECORDS_PATH, 'r', encoding='utf-8') as f:
            data = _json.load(f)
            return data if isinstance(data, list) else []
    except Exception:
        return []

def _save_records(recs):
    try:
        with open(RECORDS_PATH, 'w', encoding='utf-8') as f:
            _json.dump(recs, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print("⚠ record save error:", e)

class DailyRecord(BaseModel):
    date:       str
    worker:     str = "unknown"
    worker_id:  str = ""
    camp:       str = ""
    company:    str = ""
    dels:       int = 0
    totJ:       float = 0.0
    totKc:      float = 0.0
    maxWsi:     float = 0.0
    shiftSec:   int = 0
    deliveries: list = []
    manualCalc: dict = {}
    boxStatus:  list = []
    savedAt:    str = "" 

@app.post("/records")
def save_record(rec: DailyRecord):
    recs  = _load_records()
    ident = rec.worker_id or rec.worker
    # Replace any existing record for the same day + worker (last write wins)
    recs = [r for r in recs
            if not (r.get("date") == rec.date
                    and (r.get("worker_id") or r.get("worker")) == ident)]
    payload = rec.dict()
    payload["savedAt"] = payload.get("savedAt") or datetime.now().isoformat()
    recs.insert(0, payload)
    _save_records(recs)
    return {"success": True, "date": rec.date, "count": len(recs)}

@app.get("/records")
def get_records(date: str = None, worker_id: str = None):
    recs = _load_records()
    if date:
        recs = [r for r in recs if r.get("date") == date]
    if worker_id:
        recs = [r for r in recs if (r.get("worker_id") or "") == worker_id]
    recs.sort(key=lambda r: r.get("date", ""), reverse=True)
    return recs


#alerts
ALERTS_PATH = os.path.join(SAVE_PATH, 'risk_alerts.json')

def _load_alerts():
    try:
        with open(ALERTS_PATH, 'r', encoding='utf-8') as f:
            data = _json.load(f)
            return data if isinstance(data, list) else []
    except Exception:
        return []

def _save_alerts(arr):
    try: 
        if len(arr) > 1000: arr = arr[:1000]
        with open(ALERTS_PATH, 'w', encoding='utf-8') as f:
            _json.dump(arr, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print("⚠ alert save error:", e)

@app.post("/alerts")
def post_alert(payload: dict):
    arr = _load_alerts()
    payload["received_at"] = datetime.now().isoformat()
    arr.insert(0, payload)
    _save_alerts(arr)
    return {"success": True, "count": len(arr)}

@app.get("/alerts")
def get_alerts(worker_id: str = None, since: str = None, limit: int = 100):
    arr = _load_alerts()
    if worker_id:
        arr = [a for a in arr if (a.get("worker_id") or "") == worker_id]
    if since:
        arr = [a for a in arr if (a.get("ts") or "") >= since]
    return arr[:limit]


#deliveries
DELIVERIES_PATH = os.path.join(SAVE_PATH, 'delivery_datasets.json')

BUILDINGS_REF = {
    "백년관":      {"lat":37.337331,"lng":127.265474,"delivery_mode":"floor_delivery","max_floor":11,"has_elevator":True},
    "어문학관":    {"lat":37.338494,"lng":127.273818,"delivery_mode":"floor_delivery","max_floor":5, "has_elevator":False},
    "공학관":      {"lat":37.337721,"lng":127.267763,"delivery_mode":"mixed",          "max_floor":5, "has_elevator":False},
    "기숙사_D동":  {"lat":37.333698,"lng":127.262573,"delivery_mode":"storage",         "max_floor":4, "has_elevator":False},
    "기숙사_E동":  {"lat":37.333157,"lng":127.261437,"delivery_mode":"storage",         "max_floor":4, "has_elevator":False},
    "기숙사_A동":  {"lat":37.334453,"lng":127.262710,"delivery_mode":"storage",         "max_floor":8, "has_elevator":True},
    "기숙사_C동":  {"lat":37.335178,"lng":127.263631,"delivery_mode":"storage",         "max_floor":8, "has_elevator":True},
    "기숙사_B동":  {"lat":37.335130,"lng":127.262692,"delivery_mode":"storage",         "max_floor":8, "has_elevator":True},
    "자연과학관":  {"lat":37.338880,"lng":127.266147,"delivery_mode":"floor_delivery","max_floor":5, "has_elevator":True},
    "학생회관":    {"lat":37.337189,"lng":127.269769,"delivery_mode":"storage",         "max_floor":4, "has_elevator":False},
    "인문경상관":  {"lat":37.339748,"lng":127.274560,"delivery_mode":"storage",         "max_floor":4, "has_elevator":False},
    "창업보육센터":{"lat":37.339062,"lng":127.267414,"delivery_mode":"storage",         "max_floor":4, "has_elevator":False},
    "교양관":      {"lat":37.339740,"lng":127.272058,"delivery_mode":"storage",         "max_floor":5, "has_elevator":True},
    "후생복지관":  {"lat":37.337753,"lng":127.268621,"delivery_mode":"storage",         "max_floor":4, "has_elevator":False},
   
    "성안원룸":          {"lat":37.336538,"lng":127.254020,"delivery_mode":"floor_delivery","max_floor":3, "has_elevator":False,"off_campus":True},
    "왕산오피스텔":      {"lat":37.336434,"lng":127.254085,"delivery_mode":"floor_delivery","max_floor":3, "has_elevator":False,"off_campus":True},
    "그린빌":            {"lat":37.336532,"lng":127.253795,"delivery_mode":"floor_delivery","max_floor":4, "has_elevator":False,"off_campus":True},
    "로즈빌":            {"lat":37.336640,"lng":127.253860,"delivery_mode":"floor_delivery","max_floor":3, "has_elevator":False,"off_campus":True},
    "애버빌":            {"lat":37.336562,"lng":127.253566,"delivery_mode":"floor_delivery","max_floor":4, "has_elevator":False,"off_campus":True},
    "쉐르빌":            {"lat":37.336310,"lng":127.254403,"delivery_mode":"floor_delivery","max_floor":4, "has_elevator":False,"off_campus":True},
    "동일빌딩":          {"lat":37.335487,"lng":127.253816,"delivery_mode":"floor_delivery","max_floor":4, "has_elevator":False,"off_campus":True},
    "동일하이빌":        {"lat":37.335457,"lng":127.253949,"delivery_mode":"floor_delivery","max_floor":2, "has_elevator":False,"off_campus":True},
    "스위첸KCC 101동":   {"lat":37.334287,"lng":127.254475,"delivery_mode":"floor_delivery","max_floor":15,"has_elevator":True, "off_campus":True},
    "스위첸KCC 102동":   {"lat":37.334378,"lng":127.254030,"delivery_mode":"floor_delivery","max_floor":15,"has_elevator":True, "off_campus":True},
    "스위첸KCC 103동":   {"lat":37.334478,"lng":127.253599,"delivery_mode":"floor_delivery","max_floor":15,"has_elevator":True, "off_campus":True},
    "스위첸KCC 104동":   {"lat":37.335036,"lng":127.253818,"delivery_mode":"floor_delivery","max_floor":15,"has_elevator":True, "off_campus":True},
    "스위첸KCC 105동":   {"lat":37.334882,"lng":127.254246,"delivery_mode":"floor_delivery","max_floor":15,"has_elevator":True, "off_campus":True},
    "스위첸KCC 106동":   {"lat":37.334827,"lng":127.254680,"delivery_mode":"floor_delivery","max_floor":15,"has_elevator":True, "off_campus":True},
    "훕스테이":          {"lat":37.335403,"lng":127.254124,"delivery_mode":"floor_delivery","max_floor":4, "has_elevator":False,"off_campus":True},
    "하우스블라썸":      {"lat":37.335383,"lng":127.254174,"delivery_mode":"floor_delivery","max_floor":4, "has_elevator":False,"off_campus":True},
}
 
_SUBGROUP_OF = {
    "성안원룸":"SG_C02_WANGSAN","왕산오피스텔":"SG_C02_WANGSAN","그린빌":"SG_C02_WANGSAN",
    "로즈빌":"SG_C02_WANGSAN","애버빌":"SG_C02_WANGSAN","쉐르빌":"SG_C02_WANGSAN",
    "훕스테이":"SG_B01_OEDAERO","하우스블라썸":"SG_B01_OEDAERO","동일빌딩":"SG_B01_OEDAERO","동일하이빌":"SG_B01_OEDAERO",
    "스위첸KCC 101동":"SG_B01_SWITCHEN","스위첸KCC 102동":"SG_B01_SWITCHEN","스위첸KCC 103동":"SG_B01_SWITCHEN",
    "스위첸KCC 104동":"SG_B01_SWITCHEN","스위첸KCC 105동":"SG_B01_SWITCHEN","스위첸KCC 106동":"SG_B01_SWITCHEN",
}
for _b, _sg in _SUBGROUP_OF.items():
    if _b in BUILDINGS_REF:
        BUILDINGS_REF[_b]["subgroup"] = _sg


def _load_deliveries():
    try:
        with open(DELIVERIES_PATH, 'r', encoding='utf-8') as f:
            data = _json.load(f)
            return data if isinstance(data, dict) else {}
    except Exception:
        return {}

def _ds_key(company, date):
    return f"{company}__{date}"

@app.get("/deliveries")
def get_deliveries(company: str = None, date: str = None):
    """
    GET /deliveries                       → list available datasets (keys + counts)
    GET /deliveries?company=쿠팡&date=2026-05-28 → that day's box list
    """
    store = _load_deliveries()
    if not company or not date:
        # index of what's loaded
        idx = []
        for k, recs in store.items():
            comp, _, dt = k.partition("__")
            idx.append({"company": comp, "date": dt,
                        "boxes": len(recs),
                        "buildings": len({r.get("building") for r in recs})})
        idx.sort(key=lambda x: (x["company"], x["date"]))
        return {"datasets": idx, "buildings_ref": BUILDINGS_REF}
    recs = store.get(_ds_key(company, date), [])
    out = []
    for r in recs:
        b = BUILDINGS_REF.get(r.get("building"), {})
        out.append({**b, **r})
    return {"company": company, "date": date, "boxes": out,
            "buildings_ref": BUILDINGS_REF}

@app.post("/deliveries/bulk")
def put_deliveries(payload: dict):
    """
    Replace one dataset. Body: {company, date, boxes:[...]}.
    Used by the ingest cell, but also callable directly for quick fixes.
    """
    company = payload.get("company"); date = payload.get("date")
    boxes   = payload.get("boxes", [])
    if not company or not date:
        raise HTTPException(400, "company and date required")
    store = _load_deliveries()
    store[_ds_key(company, date)] = boxes
    try:
        with open(DELIVERIES_PATH, 'w', encoding='utf-8') as f:
            _json.dump(store, f, ensure_ascii=False, indent=2)
    except Exception as e:
        raise HTTPException(500, f"save failed: {e}")
    return {"success": True, "company": company, "date": date, "boxes": len(boxes)}


import threading, time, requests, pandas as pd

def _server_already_up():
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=1.5)
        return r.status_code == 200
    except Exception:
        return False

try:
    ngrok.kill()
except Exception:
    pass

if _server_already_up():
    print("ℹ️  uvicorn 서버가 이미 떠있음 — 재시작 건너뜀.")
else:
    def _run_server():
        import asyncio as _aio
        loop = _aio.new_event_loop()
        _aio.set_event_loop(loop)
        cfg = uvicorn.Config(app=app, host="0.0.0.0", port=8000,
                             log_level="warning", loop="asyncio")
        srv = uvicorn.Server(cfg)
        loop.run_until_complete(srv.serve())

    _t = threading.Thread(target=_run_server, daemon=True, name="uvicorn-bg")
    _t.start()
 
    _ok = False
    for _ in range(50):
        time.sleep(0.5)
        if _server_already_up():
            _ok = True
            break
    if not _ok:
        raise RuntimeError("❌ uvicorn 서버가 25초 안에 뜨지 않았습니다. "
                           "런타임을 재시작 후 다시 시도하세요.")
    print("✅ uvicorn 서버 가동 완료 (백그라운드 · port 8000)")


ngrok.set_auth_token("3CFA42fzyX2OvMiReUvvzUycUag_7ibjayh9nBXFLV1XAwWto")    
tunnel     = ngrok.connect(8000, bind_tls=True)
public_url = tunnel.public_url
print("="*55)
print(f" App URL  : {public_url}/app")
print(f" Heatmap  : {public_url}/heatmap")
print(f" Admin    : {public_url}/dashboard")
print(f" API Docs : {public_url}/docs")
print("="*55)
print(f"\n📱 휴대전화에서 위 App URL 을 여세요.")
print(f"   설정 → 서버 URL 에 다음을 저장:  {public_url}\n")


 
SERVER = "http://127.0.0.1:8000"   
 
FILES = {
    ("쿠팡", "2026-05-28"): ["/content/drive/MyDrive/배송정보_쿠팡_0528.xlsx"],
    ("쿠팡", "2026-05-29"): ["/content/drive/MyDrive/배송정보_쿠팡_0529.xlsx"],
    ("쿠팡", "2026-06-01"): ["/content/drive/MyDrive/배송정보_쿠팡_0601.xlsx"],
    ("쿠팡", "2026-06-02"): [
        "/content/drive/MyDrive/배송정보_쿠팡_0602.xlsx",          # 교내 (한국외대 캠퍼스)
        "/content/drive/MyDrive/배송정보_410C02_410B01.xlsx",      # 교외 (왕산리 자취촌)
    ],
}

def _clean_bool(v):
    return str(v).strip().upper() in ("TRUE","1","Y","YES","O","T")

import re as _re
 
_BLDG_PATTERNS = [
    (_re.compile(r"스위첸kcc?\s*101동"), "스위첸KCC 101동"),
    (_re.compile(r"스위첸kcc?\s*102동"), "스위첸KCC 102동"),
    (_re.compile(r"스위첸kcc?\s*103동"), "스위첸KCC 103동"),
    (_re.compile(r"스위첸kcc?\s*104동"), "스위첸KCC 104동"),
    (_re.compile(r"스위첸kcc?\s*105동"), "스위첸KCC 105동"),
    (_re.compile(r"스위첸kcc?\s*106동"), "스위첸KCC 106동"),
    (_re.compile(r"성안원룸"),     "성안원룸"),
    (_re.compile(r"왕산오피스텔"), "왕산오피스텔"),
    (_re.compile(r"그린빌"),       "그린빌"),
    (_re.compile(r"로즈빌"),       "로즈빌"),
    (_re.compile(r"애버빌"),       "애버빌"),
    (_re.compile(r"쉐르빌"),       "쉐르빌"),
    (_re.compile(r"동일하이빌"),   "동일하이빌"),
    (_re.compile(r"동일빌딩"),     "동일빌딩"),
    (_re.compile(r"훕스테이"),     "훕스테이"),
    (_re.compile(r"하우스블라썸"), "하우스블라썸"),
    (_re.compile(r"하나파크빌"),   "하나파크빌"),
    (_re.compile(r"파인빌"),       "파인빌"),
    (_re.compile(r"화전빌라"),     "화전빌라"),
    (_re.compile(r"이츠빌리지"),   "이츠빌리지"),
    (_re.compile(r"힐링타운빌"),   "힐링타운빌"),
    (_re.compile(r"노블레스빌"),   "노블레스빌"),
    (_re.compile(r"몬테왕"),       "몬테왕"),
]

def addr_to_building(addr_raw):
    """Map free-form unit address to canonical building name; '' if no match."""
    if not addr_raw: return ""
    flat = _re.sub(r"\s+", "", str(addr_raw).lower())
    for pat, name in _BLDG_PATTERNS:
        if pat.search(flat): return name
    return ""

def _is_road_address(s):
    """Heuristic: if the `building` column actually contains a road address
    (starts with 처인구 / contains 번길 / 대로 / 길), treat it as off-campus
    and re-derive the building name from the unit address."""
    if not s: return False
    s = str(s)
    if "처인구" in s: return True
    if any(k in s for k in ("번길", "대로", "외대로", "백옥대로")): return True
    return False

def _read_xlsx(path): 
    try:
        df = pd.read_excel(path, sheet_name="deliveries", dtype=str)
    except Exception:
        df = pd.read_excel(path, sheet_name=0, dtype=str)
    df = df.dropna(subset=["delivery_id"], how="any")
    boxes = []
    for _, row in df.iterrows():
        raw_bldg = str(row.get("building","")).strip()
        addr     = str(row.get("address","")).strip() 
        if _is_road_address(raw_bldg):
            canonical = addr_to_building(addr) or addr_to_building(raw_bldg)
            if not canonical: 
                canonical = raw_bldg
            bldg = canonical
        else:
            bldg = raw_bldg
        if not bldg or bldg.lower() == "nan":
            continue
        box = {
            "id":          str(row.get("delivery_id","")).strip(),
            "building":    bldg,
            "addr":        addr,
            "floor":       int(float(row.get("floor") or 1)),
            "hasElevator": _clean_bool(row.get("has_elevator")),
            "pkgCount":    int(float(row.get("pkg_count") or 1)),
            "pkgWeightKg": float(row.get("pkg_weight_kg") or 0) or 1.0,
        }
        for k in ["tracking_no","route_no","pkg_type","door_code","req_note","delivery_date"]:
            val = row.get(k)
            if val is not None and str(val).strip().lower() not in ("","nan"):
                box[k] = str(val).strip()
        boxes.append(box)
    return boxes

ok = fail = 0
print("📥 엑셀 적재 시작…\n")
for (company, date), paths in FILES.items(): 
    if isinstance(paths, str): paths = [paths]
    merged = []
    sources = []
    missing = []
    for p in paths:
        try:
            rows = _read_xlsx(p)
            merged.extend(rows) 
            import os as _os
            sources.append(f"{_os.path.basename(p)} ({len(rows)})")
        except FileNotFoundError:
            missing.append(_os.path.basename(p))
        except Exception as e:
            print(f"   ❌ {company} {date} [{p}]: {type(e).__name__} {e}")
            fail += 1
    if not merged:
        print(f"   ⏭️  {company} {date}: 모든 파일 없음 → 건너뜀  (missing: {missing})")
        fail += 1
        continue
    try:
        r = requests.post(f"{SERVER}/deliveries/bulk",
                          json={"company": company, "date": date, "boxes": merged},
                          timeout=20)
        r.raise_for_status()
        src_str = " + ".join(sources)
        miss_str = f"  (missing: {missing})" if missing else ""
        print(f"   ✅ {company} {date}: {len(merged)}박스 = {src_str}{miss_str}")
        ok += 1
    except Exception as e:
        print(f"   ❌ {company} {date} POST: {type(e).__name__} {e}")
        fail += 1

print("\n— 현재 서버에 로드된 데이터셋 —")
try:
    idx = requests.get(f"{SERVER}/deliveries", timeout=10).json()
    for d in idx.get("datasets", []):
        print(f"   {d['company']:5s} {d['date']}  ·  {d['boxes']}박스 / {d['buildings']}개 건물")
    if not idx.get("datasets"):
        print("   (아직 적재된 데이터 없음 — 위 FILES 경로 확인)")
except Exception as e:
    print("   index 조회 실패:", e)
print(f"\n완료: {ok}개 성공, {fail}개 건너뜀/실패")
print(f"\n🔗 휴대전화 접속 주소:  {public_url}/app")


In [ ]:
#DELIVERIES INGEST
import pandas as pd, requests, json as J, re

SERVER = "http://127.0.0.1:8000"

FILES = {
    ("쿠팡", "2026-05-28"): "/content/drive/MyDrive/배송정보_쿠팡_0528.xlsx",
    ("쿠팡", "2026-05-29"): "/content/drive/MyDrive/배송정보_쿠팡_0529.xlsx",
    ("쿠팡", "2026-06-01"): "/content/drive/MyDrive/배송정보_쿠팡_0601.xlsx",
    ("쿠팡", "2026-06-02"): "/content/drive/MyDrive/배송정보_쿠팡_0602.xlsx",
}

import sys
try:
    _hc = requests.get(SERVER + "/health", timeout=3)
    if _hc.status_code != 200:
        raise RuntimeError(f"health check returned {_hc.status_code}")
    print(f"서버 응답 확인됨 ({SERVER})\n")
except Exception as _e:
    print("━"*60)
    print("서버에 연결할 수 없습니다. 위 Cell 7(FastAPI 서버 셀)을 먼저 실행하세요.")
    print(f"   에러: {type(_e).__name__}: {_e}")
    print("━"*60)
    raise SystemExit(0)
 
_SWITCHEN_RE = re.compile(r"스위첸[\s_·]*(?:kcc)?[\s_·]*(\d{2,3})\s*동", re.I)
def normalize_building(name):
    name = str(name or "").strip()
    m = _SWITCHEN_RE.search(name)
    if m:
        return f"스위첸KCC {m.group(1)}동"
    if name == "에버빌":
        return "애버빌"
    return name
 
_SUBGROUP_OF = {
    "성안원룸":"SG_C02_WANGSAN","왕산오피스텔":"SG_C02_WANGSAN","그린빌":"SG_C02_WANGSAN",
    "로즈빌":"SG_C02_WANGSAN","애버빌":"SG_C02_WANGSAN","쉐르빌":"SG_C02_WANGSAN",
    "훕스테이":"SG_B01_OEDAERO","하우스블라썸":"SG_B01_OEDAERO","동일빌딩":"SG_B01_OEDAERO","동일하이빌":"SG_B01_OEDAERO",
    "스위첸KCC 101동":"SG_B01_SWITCHEN","스위첸KCC 102동":"SG_B01_SWITCHEN","스위첸KCC 103동":"SG_B01_SWITCHEN",
    "스위첸KCC 104동":"SG_B01_SWITCHEN","스위첸KCC 105동":"SG_B01_SWITCHEN","스위첸KCC 106동":"SG_B01_SWITCHEN",
}

OPTIONAL = ["tracking_no","route_no","pkg_type","door_code","req_note","delivery_date"]

def _clean_bool(v):
    return str(v).strip().upper() in ("TRUE","1","Y","YES","O","T")

def _read_sheet(path, names):
    """엑셀에서 후보 시트명 중 존재하는 첫 시트를 읽는다. 없으면 첫 시트."""
    xl = pd.ExcelFile(path)
    for n in names:
        if n in xl.sheet_names:
            return pd.read_excel(xl, sheet_name=n, dtype=str)
    return pd.read_excel(xl, sheet_name=0, dtype=str)

def read_buildings(path):
    """buildings 시트(있으면) → {정규화건물명: {lat,lng,delivery_mode,max_floor,has_elevator}}."""
    try:
        xl = pd.ExcelFile(path)
        if "buildings" not in xl.sheet_names:
            return {}
        b = pd.read_excel(xl, sheet_name="buildings", dtype=str)
    except Exception:
        return {}
    ref = {}
    for _, r in b.iterrows():
        nm = normalize_building(r.get("building"))
        if not nm or nm.lower() == "nan":
            continue
        meta = {}
        try: meta["lat"] = float(r.get("lat"))
        except Exception: pass
        try: meta["lng"] = float(r.get("lng"))
        except Exception: pass
        if str(r.get("delivery_mode") or "").strip():
            meta["delivery_mode"] = str(r.get("delivery_mode")).strip()
        try: meta["max_floor"] = int(float(r.get("max_floor")))
        except Exception: pass
        meta["default_has_elevator"] = _clean_bool(r.get("default_has_elevator"))
        if nm in _SUBGROUP_OF:
            meta["subgroup"] = _SUBGROUP_OF[nm]
        ref[nm] = meta
    return ref

def read_one(path):
    df = _read_sheet(path, ["deliveries", "Sheet1"])
    df = df.dropna(subset=["delivery_id", "building"], how="any")
    bref = read_buildings(path)
    boxes = []
    for _, row in df.iterrows():
        raw = str(row.get("building", "")).strip()
        if not raw or raw.lower() == "nan":
            continue
        bldg = normalize_building(raw)
        meta = bref.get(bldg, {})
        box = {
            "id":          str(row.get("delivery_id", "")).strip(),
            "building":    bldg,                                  # 정규화된 표준 건물명
            "addr":        str(row.get("address", "")).strip(),
            "floor":       int(float(row.get("floor") or 1)),
            "hasElevator": _clean_bool(row.get("has_elevator")),
            "pkgCount":    int(float(row.get("pkg_count") or 1)),
            "pkgWeightKg": float(row.get("pkg_weight_kg") or 0) or 1.0,
            "subgroup":    _SUBGROUP_OF.get(bldg, meta.get("subgroup", "")),
        }
        # buildings 시트의 좌표/모드/층 정보를 박스에 병합(앱·히트맵 표시용)
        for k in ("lat", "lng", "delivery_mode", "max_floor"):
            if k in meta:
                box[k] = meta[k]
        for k in OPTIONAL:
            val = row.get(k)
            if val is not None and str(val).strip().lower() not in ("", "nan"):
                box[k] = str(val).strip()
        boxes.append(box)
    return boxes

ok, fail = 0, 0
for (company, date), path in FILES.items():
    try:
        boxes = read_one(path)
        r = requests.post(f"{SERVER}/deliveries/bulk",
                          json={"company": company, "date": date, "boxes": boxes},
                          timeout=15)
        r.raise_for_status()
        nsub = len({b["subgroup"] for b in boxes if b.get("subgroup")})
        print(f"{company} {date}: {len(boxes)}박스 전송 (서브그룹 {nsub}개)")
        ok += 1
    except FileNotFoundError:
        print(f" {company} {date}: 파일 없음 → 건너뜀 ({path})")
        fail += 1
    except Exception as e:
        print(f" {company} {date}: {type(e).__name__} {e}")
        fail += 1

print("\n— 현재 서버에 로드된 데이터셋 —")
try:
    idx = requests.get(f"{SERVER}/deliveries", timeout=10).json()
    for d in idx.get("datasets", []):
        print(f"   {d['company']:5s} {d['date']}  ·  {d['boxes']}박스 / {d['buildings']}개 건물")
except Exception as e:
    print("index 조회 실패:", e)
print(f"\n완료: {ok}개 성공, {fail}개 건너뜀/실패")
